# Deferring measurements

**Download this notebook - {nb-download}`deferred_measurement.ipynb`**

In this example we will illustrate the `Measurement` type and why the ability to defer measurements can be important for performance.

In [1]:
from guppylang import guppy
from guppylang.std.builtins import array, owned
from guppylang.std.quantum import qubit, measure
from guppylang.std.qsystem import lazy_measure
from guppylang.std.platform import result

from typing import no_type_check

In previous versions of Guppy, the `measure` function used to return a `bool` directly, which implied a measurement was being performed immediately. However, this was hiding what was actually happening on the hardware, which is that a measurement request is being made when the function is called, and the physical measurement can be deferred until it is actually required.

Now `measure` (and adjacent functions such as `measure_array`) return a `Measurement` type, which is a handle to the measurement request. In order to block on the request and wait to receive a bool, the `read()` method should be used. This allows the runtime to defer measurements until they are actually needed, which can give more opportunities for parallelism and therefore better performance. Exposing this behaviour allows you to make more informed decisions about how to structure your program for better performance.

Consider the following program, where you might be tempted to use the output of `measure` directly in `result`:

In [2]:
@guppy
@no_type_check
def output_result(qs: array[qubit, 10] @ owned) -> None:
    for q in qs:
        result("t", measure(q))
    # [... more quantum operations  ...]


output_result.check()

Error: Invalid call of overloaded function (at <In[2]>:5:15)
  | 
3 | def output_result(qs: array[qubit, 10] @ owned) -> None:
4 |     for q in qs:
5 |         result("t", measure(q))
  |                ^^^^^^^^^^^^^^^ No variant of overloaded function `result` takes arguments
  |                                `str`, `Measurement`

Note: Available overloads are:
  def result(tag: str @comptime, value: int) -> None
  def result(tag: str @comptime, value: nat) -> None
  def result(tag: str @comptime, value: bool) -> None
  def result(tag: str @comptime, value: float) -> None
  def result(tag: str @comptime, value: array[int, n]) -> None
  def result(tag: str @comptime, value: array[nat, n]) -> None
  def result(tag: str @comptime, value: array[bool, n]) -> None
  def result(tag: str @comptime, value: array[float, n]) -> None

Guppy compilation failed due to 1 previous error


Simply replacing the method results in an error because `result` expects a `bool`. In order to obtain it, we have to explicitly use `read()` on the value of type `Measurement`. This will block until the physical measurement is completed:

In [3]:
@guppy
@no_type_check
def output_result(qs: array[qubit, 10] @ owned) -> None:
    for q in qs:
        result("t", lazy_measure(q).read())
    # [... more quantum operations  ...]

output_result.check()

The program now type-checks and we can see where exactly the `read()` happens, rather than it being done implicitly. Therefore we can try to move it further down the program, which allows for flexibility in when the quantum operations before the blocking read are performed:

In [4]:
@guppy
@no_type_check
def output_result_improved(qs: array[qubit, 10] @ owned) -> None:
    ms = array(measure(q) for q in qs)
    # [... more quantum operations  ...]
    for m in ms:
        # This `read` call only blocks execution when we are at the end of the program 
        result("t", m.read()) 

output_result_improved.check()


For convenience, `__bool__` is implemented on the `Measurement` type as syntactic sugar for `read()`, so it is called automatically in conditionals, for example:

In [5]:
@guppy
@no_type_check
def lazy_conditional(q: qubit @ owned) -> None:
    if measure(q):
        result("t", 1)
    else:
        result("t", 0)

lazy_conditional.check()